# Prefill vs Decode: the Latency Split, in Numbers**GDE #3 — vLLM serving internals.** Companion notebook for"Prefill vs Decode: Why Splitting Them Wins" (Nov 13, 2026).Anchor model: an 8B-class dense model (32 layers, 8 KV heads, head dim 128, bf16)on one H100 SXM (990 BF16 TFLOPS peak, 3.35 TB/s HBM). Every number below iscomputed, not quoted — the paper/vendor figures it cites are flagged in the post.

In [ ]:
# Cell 1 — KV bytes per token, and what moving them costs on each link.
LAYERS, KV_HEADS, HEAD_DIM, BYTES = 32, 8, 128, 2
PARAMS = 8e9
kv_per_token = LAYERS * 2 * KV_HEADS * HEAD_DIM * BYTES
assert kv_per_token == 131072, kv_per_token          # 128 KB per token
print(f"KV cache per token: {kv_per_token/1024:.0f} KB")

links = {"NVLink (in-node)": 450e9, "InfiniBand 200 Gb/s": 25e9,
         "100 GbE": 12.5e9, "25 GbE": 3.125e9}
for name, bw in links.items():
    us = kv_per_token / bw * 1e6
    print(f"{name:22s} {us:6.2f} us/token")
# Prefill math per token at peak: 2*params FLOPs / 990 TFLOPS
prefill_us_peak = 2 * PARAMS / 990e12 * 1e6
print(f"Prefill math per token (peak): {prefill_us_peak:.1f} us")
assert abs(prefill_us_peak - 16.2) < 0.5
# A 2,000-token prompt's KV cache is 256 MB — and its transfer time:
for name, bw in links.items():
    ms = 2000 * kv_per_token / bw * 1000
    print(f"2K-prompt KV transfer over {name:22s} {ms:6.2f} ms")
assert abs(2000 * kv_per_token / 12.5e9 * 1000 - 20.97) < 0.1

In [ ]:
# Cell 2 — Prefill vs decode cost model: what each phase actually costs.
# Prefill: compute-bound. Decode step: read all weights + every live KV cache.
H100_TFLOPS, H100_BW, MFU = 990e12, 3.35e12, 0.40
weights_gb = PARAMS * 2 / 1e9                       # 16 GB
prefill_us = 2 * PARAMS / H100_TFLOPS / MFU * 1e6   # ~40 us/token at 40% MFU
for prompt in [512, 2000, 8000, 32000]:
    print(f"prefill {prompt:6d} tokens: {prompt*prefill_us/1000:7.1f} ms  "
          f"(KV produced: {prompt*kv_per_token/1e9:.2f} GB)")
def decode_step_ms(batch, ctx_tokens):
    kv_gb = batch * ctx_tokens * kv_per_token / 1e9
    return (weights_gb + kv_gb) / (H100_BW / 1e9) * 1000
for b, ctx in [(1, 2000), (32, 2000), (128, 2000), (32, 32000)]:
    print(f"decode step batch={b:3d} ctx={ctx:6d}: {decode_step_ms(b, ctx):5.1f} ms")
assert 6.0 < decode_step_ms(32, 2000) < 8.0
assert abs(decode_step_ms(1, 2000) - 4.9) < 0.3
# Takeaway: prefill time grows with PROMPT length; decode step grows with BATCH x CONTEXT.

In [ ]:
# Cell 3 — Interference simulation: one 8,000-token prompt lands mid-stream.
# Colocated whole prefill freezes every decoding user; 128-token chunks cap the
# freeze; disaggregation removes it entirely (decode GPU never prefills).
import numpy as np
n_steps = 400
base = decode_step_ms(32, 2000)                     # ~7.3 ms decode step
whole_stall = 8000 * prefill_us / 1000              # ~323 ms
chunk_stall = 128 * prefill_us / 1000               # ~5.2 ms
def simulate(policy):
    itl, remaining = [], 8000
    for s in range(n_steps):
        if policy == "colocated" and s == 120 and remaining:
            itl.append(base + whole_stall); remaining = 0
        elif policy == "chunked" and s % 8 == 0 and remaining:
            itl.append(base + chunk_stall); remaining -= 128
        else:
            itl.append(base)
    return np.array(itl)
for p in ["colocated", "chunked", "disaggregated"]:
    v = simulate(p)
    print(f"{p:14s} max ITL {np.max(v):6.1f} ms   p99 {np.percentile(v,99):5.1f} ms   "
          f"fraction of steps > 50 ms SLO: {(v > 50).mean():.3f}")
    if p == "colocated":
        assert np.max(v) > 300
    else:
        assert np.percentile(v, 99) < 15
# The average TPOT barely moves in any policy — the freeze hides in the tail.
# That is why vLLM's docs frame disaggregation as a TAIL-latency tool, not a
# throughput tool: "Disaggregated prefill DOES NOT improve throughput."

In [ ]:
# Cell 4 — Pool sizing with Little's law: how many prefill vs decode GPUs?
# Arrival rate lam (req/s), prefill service time S_p, decode service time S_d
# (decode holds a GPU slot for the whole output, ~output_len * step time).
lam = 20.0                       # requests/second
S_p = 2000 * prefill_us / 1e6    # seconds of prefill per request (2K prompt)
out_tokens, step_ms = 200, decode_step_ms(32, 2000)
S_d = out_tokens * step_ms / 1000
for util in [0.7, 0.8]:
    n_p = lam * S_p / util
    n_d = lam * S_d / util
    print(f"at {util*100:.0f}% util: prefill GPUs {n_p:5.1f}   decode GPUs {n_d:5.1f}   "
          f"ratio P:D = 1:{n_d/n_p:.1f}")
# DistServe's measured point: 1.6 req/s colocated vs 10 req/s on 2 prefill + 1
# decode GPUs = 3.3 req/s/GPU, a 2.1x goodput-per-GPU win — the split pays when
# the two phases want different GPU counts, which Little's law predicts.
assert abs(lam * S_p / 0.8 - 2.0) < 0.3

In [ ]:
# Cell 5 — The decision rule: when does the split beat chunked prefill?
# Split wins when the KV-transfer tax is smaller than the stall it removes.
# Stall removed ~= whole-prefill time; tax = KV bytes / link bandwidth.
import numpy as np
for prompt in [512, 2000, 8000, 32000]:
    stall = prompt * prefill_us / 1000
    tax_100g = prompt * kv_per_token / 12.5e9 * 1000
    tax_25g = prompt * kv_per_token / 3.125e9 * 1000
    print(f"{prompt:6d}-token prompt: stall removed {stall:7.1f} ms | "
          f"transfer tax 100GbE {tax_100g:6.1f} ms | 25GbE {tax_25g:6.1f} ms | "
          f"net win on 100GbE: {stall - tax_100g:6.1f} ms")
# On 100 GbE the transfer is ~26% of the stall at every prompt length, so the
# split wins on latency at any scale — IF the pool ratio matches traffic.
# On 25 GbE the tax EXCEEDS the stall (41.9 us/token vs ~40 us/token of math):
# splitting across slow Ethernet makes TTFT worse, not better. The interconnect
# is the decision, not the idea.
assert 2000 * kv_per_token / 3.125e9 * 1000 > 2000 * prefill_us / 1000

## What to take away1. Prefill is compute-bound, decode is memory-bandwidth-bound — one schedule cannot serve both.2. A whole-prompt prefill freezes every decoding user for hundreds of ms; chunked prefill caps the freeze, disaggregation removes it.3. The KV transfer costs 128 KB/token for an 8B model: 0.3 µs/token on NVLink, 10.5 µs on 100 GbE, 41.9 µs on 25 GbE — the link decides whether splitting helps.4. vLLM's own docs: disaggregated prefill does **not** improve throughput. It buys separately tunable TTFT and tail ITL. Size the pools with Little's law, or the split just moves the queue.